# 02 · Crave Assist: the CraveCart RAG chat

Builds on the class `RAG_APP.ipynb` (LangChain + Chroma + Groq + Gradio) and adds:

- **Follow-up rewriting:** each question plus the chat history becomes one standalone English search query, so follow-ups and Hindi/Hinglish questions retrieve the right passages.
- **Citations:** answers mark the passages they use as [1], [2]…, and the app shows them in a Sources panel.
- **Streaming** answers in **English, Hindi or Hinglish**, whichever the user wrote in.

**Key:** in Colab, add `GROQ_API_KEY` under 🔑 Secrets. Locally, put it in `.env`.

## Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SAVE_DB_TO_DRIVE = False  # True: keep the vector DB in Google Drive, like the class notebook

if IN_COLAB:
    if not Path("ai-engineering-projects").exists():
        !git clone --depth 1 https://github.com/C0derANANT/ai-engineering-projects.git
    %cd ai-engineering-projects/03-Food-Delivery-RAG
    %pip install -q -r requirements.txt
    if SAVE_DB_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["VECTOR_DB_DIR"] = "/content/drive/MyDrive/ai/cravecart_vector_db"
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # run from the project folder so `rag` can be imported

sys.path.insert(0, str(Path.cwd()))
print("Project folder:", Path.cwd())

if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY") or ""
    except Exception:
        print("GROQ_API_KEY not found in Colab secrets")

## 1. Connect the vector DB and the models
Builds the database if it doesn't exist yet (or if the documents changed).

In [ ]:
from rag.assistant import Assistant, get_models
from rag.config import Settings
from rag.knowledge_base import open_vector_store, search

settings = Settings()
store, db_info = open_vector_store(settings)
assistant = Assistant(settings, search_fn=lambda query, k: search(store, query, k))
models = get_models(settings)
print(f"Vector DB: {db_info.chunks} chunks")
print("Model:", models.name if models else "none (add GROQ_API_KEY)")

## 2. Query rewriting
The raw message is turned into a standalone English search query, and its language is detected.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

history = [HumanMessage(content="Can I cancel my order?"), AIMessage(content="Yes, free within 60 seconds of placing it.")]
for question, past in [("Mere order mein ek item nahi aaya, refund kaise milega?", []),
                       ("what if the restaurant had already started cooking?", history)]:
    query, language = assistant.rewrite_query(models, past, question) if models else (question, None)
    print(f"{question}\n  -> query: {query!r} · language: {language}\n")

## 3. Retrieval
The 5 closest passages for a query, with the document and section they come from.

In [ ]:
for hit in search(store, "cancellation fee after the restaurant starts cooking", 5):
    print(f"[{hit.number}] {hit.score:.0%}  {hit.label}")

## 4. Ask questions
`ask()` runs the whole pipeline (rewrite → retrieve → answer) and shows the sources.

In [ ]:
from IPython.display import Markdown, display
from rag.ui import sources_markdown

chat_history = []

def ask(question):
    reply = None
    for reply in assistant.respond(question, chat_history):
        pass  # the app streams these updates; here we just keep the last one
    chat_history.extend([{"role": "user", "content": question}, {"role": "assistant", "content": reply.text}])
    display(Markdown(f"**Q: {question}**\n\n{reply.text}\n\n---\n{sources_markdown(reply)}"))

ask("Can I cancel my order after the restaurant starts cooking? Is there a fee?")

In [ ]:
ask("And what if the delivery partner has already picked it up?")  # a follow-up

In [ ]:
ask("मेरे ऑर्डर में एक आइटम नहीं आया, रिफंड कैसे मिलेगा?")

## 5. Launch the Gradio app
In Colab this prints a public `gradio.live` link.

In [ ]:
import app  # builds the UI from app.py

if os.environ.get("SKIP_LAUNCH") != "1":
    app.launch(share=IN_COLAB, inbrowser=not IN_COLAB)